# GLDM Delta Rollover — TLT Hedge (Live)

**Strategy.** Hold a long position in **GLDM** (SPDR Gold MiniShares Trust) and hedge the rate-sensitive component of the gold exposure with a short position in **TLT** (iShares 20+ Year Treasury Bond ETF). Re-estimate the hedge ratio on a rolling window ("delta") and roll the hedge on a schedule or when drift exceeds a threshold.

**Data Source:** Live Alpha Vantage via the shared `alpha_vantage_live` module (replaces yfinance)  
**Entry Point:** `GLDM_Delta_Rollover_TLT_Hedge_Live(api_key)`  
**Docs:** https://www.alphavantage.co/documentation &nbsp;·&nbsp; https://www.alphavantage.co/premium/

---

| Section | Purpose |
|---|---|
| **§1 — Environment & Configuration** | Imports, dataclass config, logging |
| **§2 — Alpha Vantage Data Client** | Import shared live client; pull GLDM & TLT |
| **§3 — Hedge Ratio & Delta Engine** | Rolling OLS beta, hedge notionals, roll signal |
| **§4 — Live Report** | Position table, roll recommendation, CSV export |
| **§5 — Entry Point** | `GLDM_Delta_Rollover_TLT_Hedge_Live(api_key)` |

---
## 1. Environment & Configuration

In [ ]:
# =============================================================================
# §1.1 — Imports
# =============================================================================
import sys
import logging
import warnings
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Dict

import numpy as np
import pandas as pd

from alpha_vantage_live import (
    AlphaVantageConfig,
    AlphaVantageClient,
    CacheManager,
)

warnings.filterwarnings('ignore', category=FutureWarning)
logging.getLogger('urllib3').setLevel(logging.CRITICAL)

print(f'python:  {sys.version.split()[0]}')
print(f'pandas:  {pd.__version__}')
print(f'numpy:   {np.__version__}')

In [ ]:
# =============================================================================
# §1.2 — Configuration Dataclass
# =============================================================================
@dataclass
class DeltaHedgeConfig:
    """Hedge-engine config — no magic numbers in logic cells."""

    # Universe
    asset_ticker: str = 'GLDM'
    hedge_ticker: str = 'TLT'

    # Hedge engine
    beta_window: int = 60
    min_bars: int = 252
    roll_threshold_bps: float = 150.0
    roll_calendar_days: int = 21
    target_asset_notional: float = 1_000_000.0

    # Output
    output_dir: str = './delta_hedge_output'

    def av_config(self, api_key: str) -> AlphaVantageConfig:
        """Build an Alpha Vantage client config bound to this strategy."""
        return AlphaVantageConfig(api_key=api_key)

---
## 2. Alpha Vantage Data Client

The live client lives in `alpha_vantage_live.py` — a reusable module any notebook in this repo can import in place of yfinance. It uses the `TIME_SERIES_DAILY_ADJUSTED` endpoint (Premium) and falls back to `TIME_SERIES_DAILY` if the adjusted feed is gated. Parsed DataFrames are cached as parquet with a TTL.

In [ ]:
# =============================================================================
# §2.1 — Live Pull Helper
# =============================================================================
def fetch_prices(
    api_key: str,
    hedge_cfg: DeltaHedgeConfig,
) -> pd.DataFrame:
    """Return a wide DataFrame of adjusted closes for asset + hedge."""
    av_cfg = hedge_cfg.av_config(api_key)
    cache = CacheManager(av_cfg)
    client = AlphaVantageClient(av_cfg, cache)
    prices = client.download(
        [hedge_cfg.asset_ticker, hedge_cfg.hedge_ticker],
        field='AdjClose',
    )
    return prices.dropna(how='any')

---
## 3. Hedge Ratio & Delta Engine

Rolling OLS regression of **GLDM** daily log returns on **TLT** daily log returns over `beta_window` trading days. The fitted slope is the **delta** — the dollar sensitivity of GLDM to a unit move in TLT. The TLT hedge notional is

$$\text{TLT}_{\text{notional}} = -\beta_t \cdot \frac{P^{\text{GLDM}}_t}{P^{\text{TLT}}_t} \cdot N^{\text{GLDM}}$$

A **roll signal** fires when the realised hedge drift exceeds `roll_threshold_bps` or the calendar-day age exceeds `roll_calendar_days`.

In [ ]:
# =============================================================================
# §3.1 — Rolling Beta (Delta)
# =============================================================================
def rolling_beta(returns: pd.DataFrame, asset: str, hedge: str, window: int) -> pd.Series:
    """Rolling OLS slope of asset on hedge via cov/var."""
    cov = returns[asset].rolling(window).cov(returns[hedge])
    var = returns[hedge].rolling(window).var()
    return (cov / var).rename('beta')

def compute_delta_hedge(prices: pd.DataFrame, config: DeltaHedgeConfig) -> pd.DataFrame:
    """Compute the live delta, hedge notional, and roll flags."""
    asset, hedge = config.asset_ticker, config.hedge_ticker
    if len(prices) < config.min_bars:
        raise RuntimeError(
            f'Insufficient history: {len(prices)} rows < min_bars={config.min_bars}'
        )

    log_returns = np.log(prices).diff()
    beta = rolling_beta(log_returns, asset, hedge, config.beta_window)

    asset_units = config.target_asset_notional / prices[asset]
    hedge_units = -beta * asset_units * prices[asset] / prices[hedge]
    hedge_notional = hedge_units * prices[hedge]

    table = pd.DataFrame({
        f'{asset}_price': prices[asset],
        f'{hedge}_price': prices[hedge],
        'beta': beta,
        f'{asset}_units': asset_units,
        f'{hedge}_units': hedge_units,
        f'{hedge}_notional': hedge_notional,
    }).dropna()

    table['beta_drift_bps'] = (table['beta'].diff().abs() * 10_000).fillna(0.0)
    table['bars_since_roll'] = np.nan
    last_roll_idx = table.index[0]
    for idx in table.index:
        days = (idx - last_roll_idx).days
        table.loc[idx, 'bars_since_roll'] = days
        drift_trigger = table.loc[idx, 'beta_drift_bps'] > config.roll_threshold_bps
        calendar_trigger = days >= config.roll_calendar_days
        if drift_trigger or calendar_trigger:
            last_roll_idx = idx
            table.loc[idx, 'roll_signal'] = 1
        else:
            table.loc[idx, 'roll_signal'] = 0
    table['roll_signal'] = table['roll_signal'].astype(int)
    return table

---
## 4. Live Report

In [ ]:
# =============================================================================
# §4.1 — Status Banner
# =============================================================================
def print_status(config: DeltaHedgeConfig, hedge_tbl: pd.DataFrame) -> None:
    latest = hedge_tbl.iloc[-1]
    asof = hedge_tbl.index[-1].strftime('%Y-%m-%d')
    asset, hedge = config.asset_ticker, config.hedge_ticker
    print('=' * 72)
    print('  GLDM DELTA ROLLOVER — TLT HEDGE (LIVE ALPHA VANTAGE)')
    print(f'  As of {asof}   (generated {datetime.now().strftime("%Y-%m-%d %H:%M:%S")})')
    print('=' * 72)
    print(f'  {asset} price:        ${latest[f"{asset}_price"]:>12,.2f}')
    print(f'  {hedge} price:        ${latest[f"{hedge}_price"]:>12,.2f}')
    print(f'  Rolling beta ({config.beta_window}d):  {latest["beta"]:>12,.4f}')
    print(f'  Target {asset} notional: ${config.target_asset_notional:>12,.0f}')
    print(f'  {asset} units (long):  {latest[f"{asset}_units"]:>12,.2f}')
    print(f'  {hedge} units (hedge): {latest[f"{hedge}_units"]:>12,.2f}')
    print(f'  {hedge} notional:     ${latest[f"{hedge}_notional"]:>12,.0f}')
    print(f'  Beta drift (bps):     {latest["beta_drift_bps"]:>12,.1f}')
    print(f'  Roll signal today:    {"YES" if latest["roll_signal"] == 1 else "no"}')
    print('=' * 72)

In [ ]:
# =============================================================================
# §4.2 — CSV Export
# =============================================================================
def export_hedge_report(hedge_tbl: pd.DataFrame, config: DeltaHedgeConfig) -> Path:
    out_dir = Path(config.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    path = out_dir / f'GLDM_TLT_delta_hedge_{stamp}.csv'
    hedge_tbl.to_csv(path, index=True)
    latest_path = out_dir / 'GLDM_TLT_delta_hedge_latest.csv'
    hedge_tbl.to_csv(latest_path, index=True)
    print(f'  [OK] wrote {path}')
    print(f'  [OK] wrote {latest_path}')
    return path

---
## 5. Entry Point

Call `GLDM_Delta_Rollover_TLT_Hedge_Live(api_key)` with your Alpha Vantage key. Returns the latest hedge row as a `pd.Series` and writes a CSV to `./delta_hedge_output/`.

In [ ]:
# =============================================================================
# §5.1 — Main Entry Point
# =============================================================================
def GLDM_Delta_Rollover_TLT_Hedge_Live(
    api_key: str,
    config: Optional[DeltaHedgeConfig] = None,
) -> pd.Series:
    """Fetch GLDM and TLT live from Alpha Vantage and compute the delta-rollover hedge.

    Parameters
    ----------
    api_key : str
        Alpha Vantage API key (Premium recommended for adjusted series).
    config : DeltaHedgeConfig, optional
        Override default hedge-engine configuration.

    Returns
    -------
    pd.Series
        Most recent hedge row (price, beta, units, notional, roll_signal).
    """
    cfg = config or DeltaHedgeConfig()
    prices = fetch_prices(api_key, cfg)
    hedge_tbl = compute_delta_hedge(prices, cfg)
    print_status(cfg, hedge_tbl)
    export_hedge_report(hedge_tbl, cfg)
    return hedge_tbl.iloc[-1]

In [ ]:
# =============================================================================
# §5.2 — Run Live
# =============================================================================
latest = GLDM_Delta_Rollover_TLT_Hedge_Live('A9R0FMDJEHEDNWAZ')
latest